# biblia-texto-baixar.ipynb — Texto da Bíblia (WEB) pro projeto

Baixa a **World English Bible** em USFM do ebible.org, converte pra um JSON
único em `pipeline/dados_lexico/web-biblia.json`, e **confere contra o
`40_Matt_02` que já existe** antes de dar por bom.

Com isso, gerar o `roteiro_versiculos.txt` de qualquer capítulo vira uma
chamada de função — acabou a consulta capítulo a capítulo no site.

## Por que USFM (e não o PDF)

O `WEBTEXT.pdf` da página da narração serve pra ler, não pra virar dado: é de
duas colunas, e extrator de texto embaralha as palavras entre elas. O USFM
marca explicitamente parágrafo (`\p`) e poesia (`\q1`) — que é exatamente a
estrutura que o `roteiro_versiculos.txt` do projeto já tem.

## A conferência não é decorativa

O passo 4 compara o Mateus 2 recém-baixado com o arquivo que você já usou pra
gerar vídeo. **Traduções diferentes em inglês batem ~0,83 de similaridade** —
não 0,2, porque compartilham muita palavra. Por isso o limiar é 0,97: frouxo
demais e uma tradução errada passaria despercebida.

Isso importa porque o `alinhar_versiculos()` casa este texto contra a
transcrição do Whisper pra derivar o tempo de cada versículo. Texto de outra
edição degrada o alinhamento **em silêncio** — aparece só no vídeo montado.

---
*WEB: domínio público, sem restrição de copyright (Michael Paul Johnson /
eBible.org).*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — Drive e módulos                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if not PASTA_MODULOS.exists():
    raise SystemExit(f"❌ Módulos não encontrados: {PASTA_MODULOS}")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos copiados")

if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. DE ONDE BAIXAR ───────────────────────────────────────────────────────
# O ebible.org publica cada tradução como um zip de USFM. A WEB tem mais de
# uma edição; a que pareia com a narração do David Williams é a **Classic**.
#
# Candidatos tentados EM ORDEM, ficando no primeiro que responder um zip
# válido. Se todos falharem, o notebook para e te manda conferir o link --
# ele não inventa fonte alternativa.
URLS_CANDIDATAS = [
    "https://ebible.org/Scriptures/eng-web-c_usfm.zip",   # WEB Classic
    "https://ebible.org/Scriptures/eng-web_usfm.zip",     # WEB
    "https://ebible.org/Scriptures/engwebp_usfm.zip",     # WEB (protestante)
]

# Se você já sabe o link certo, cole aqui e os candidatos acima são ignorados.
# Pra achar: https://ebible.org/eng-web-c/  ->  seção de downloads.
URL_MANUAL = ""

# ── 2. ONDE SALVAR ──────────────────────────────────────────────────────────
# dados_lexico/ é onde já moram os dados de referência do projeto
# (eventos-biblicos.json, titulos-biblicos.json). O texto bíblico é a mesma
# categoria: imutável, versionado, lido por módulo -- não é planilha.
NOME_SAIDA = "web-biblia.json"

# ── 3. CONFERÊNCIA ──────────────────────────────────────────────────────────
# Capítulo já existente no projeto usado como prova de que o texto baixado é
# mesmo o da narração.
CONFERIR_CONTRA = "40_Matt_02"     # pasta em videos/
CONFERIR_SIGLA, CONFERIR_CAP = "Matt", 2

# Limiar de similaridade. Calibrado: WEB x KJV no mesmo trecho dá ~0,83, então
# qualquer coisa abaixo de 0,97 é suspeita de edição/tradução diferente.
LIMIAR = 0.97

print(f"Saída ......... dados_lexico/{NOME_SAIDA}")
print(f"Conferir ...... {CONFERIR_SIGLA} {CONFERIR_CAP} vs {CONFERIR_CONTRA}")
print(f"Limiar ........ {LIMIAR}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR                                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import biblia_livros as bl
import biblia_texto as bt

BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
PASTA_DADOS = BASE / "pipeline" / "dados_lexico"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
CAMINHO_SAIDA = PASTA_DADOS / NOME_SAIDA

TRABALHO = Path("/content/biblia_usfm"); TRABALHO.mkdir(exist_ok=True)

print(f"📖 {len(bl.LIVROS)} livros, {bl.TOTAL_CAPITULOS} capítulos esperados")
print(f"📁 {CAMINHO_SAIDA}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⬇️  1/4 — BAIXAR O USFM                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import urllib.request, zipfile, io

TAMANHO_MINIMO = 1_000_000   # o USFM da Bíblia inteira passa de 1 MB zipado

candidatos = [URL_MANUAL] if URL_MANUAL else URLS_CANDIDATAS
zip_bytes, url_usada = None, None

for url in candidatos:
    print(f"⬇️  tentando {url}")
    try:
        with urllib.request.urlopen(url, timeout=120) as r:
            dados = r.read()
    except Exception as e:
        print(f"   ✗ {type(e).__name__}: {e}")
        continue

    # Servidor pode responder 200 com página de erro. Só aceita se for zip
    # de verdade e do tamanho esperado -- senão o problema só apareceria na
    # hora de parsear, com uma mensagem que não ajuda.
    if len(dados) < TAMANHO_MINIMO:
        print(f"   ✗ veio {len(dados)/1e3:.0f} KB, esperava > {TAMANHO_MINIMO/1e6:.0f} MB")
        continue
    if not dados[:2] == b"PK":
        print("   ✗ não é um zip")
        continue

    zip_bytes, url_usada = dados, url
    print(f"   ✅ {len(dados)/1e6:.1f} MB")
    break

if zip_bytes is None:
    raise SystemExit(
        "❌ Nenhum candidato respondeu um zip USFM válido.\n"
        "   Abra https://ebible.org/eng-web-c/ , ache o link de download USFM\n"
        "   e cole em URL_MANUAL na célula de Configuração.")

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    z.extractall(TRABALHO)

arquivos = sorted(TRABALHO.rglob("*.usfm")) + sorted(TRABALHO.rglob("*.SFM"))
print(f"📄 {len(arquivos)} arquivos USFM")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📖 2/4 — PARSEAR                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

livros_lidos, ignorados = {}, []

for caminho in arquivos:
    conteudo = caminho.read_text(encoding="utf-8-sig", errors="replace")
    try:
        livro, capitulos = bt.parsear_usfm(conteudo)
    except (ValueError, KeyError) as e:
        # Deuterocanônicos e arquivos de front matter caem aqui -- o zip do
        # ebible.org traz mais coisa que os 66 livros. Não é erro.
        ignorados.append((caminho.name, str(e)[:60]))
        continue
    livros_lidos[livro.sigla] = capitulos

print(f"✅ {len(livros_lidos)} livros dos 66")
if ignorados:
    print(f"⏭️  {len(ignorados)} arquivos fora do cânone de 66 (normal)")

faltando = [l.sigla for l in bl.LIVROS if l.sigla not in livros_lidos]
if faltando:
    raise SystemExit(f"❌ Livros não encontrados no zip: {faltando}")

# Confere a contagem de capítulos livro a livro contra o cânone.
problemas = []
for livro in bl.LIVROS:
    lidos = len(livros_lidos[livro.sigla])
    if lidos != livro.capitulos:
        problemas.append(f"{livro.nome}: {lidos} capítulos, esperava {livro.capitulos}")

if problemas:
    print("\n⚠️  divergência na contagem de capítulos:")
    for p in problemas:
        print(f"   {p}")
else:
    total = sum(len(c) for c in livros_lidos.values())
    print(f"✅ {total} capítulos — bate com o cânone ({bl.TOTAL_CAPITULOS})")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 3/4 — CONFERIR CONTRA O CAPÍTULO QUE JÁ EXISTE                ║
# ║  A prova de que este texto é mesmo o da narração                  ║
# ╚══════════════════════════════════════════════════════════════════╝

caminho_existente = (BASE / "videos" / CONFERIR_CONTRA /
                     f"{CONFERIR_CONTRA}_roteiro_versiculos.txt")

if not caminho_existente.exists():
    print(f"⏭️  {caminho_existente.name} não está no Drive — conferência pulada")
else:
    texto_existente = caminho_existente.read_text(encoding="utf-8")
    texto_baixado = bt.gerar_roteiro(livros_lidos[CONFERIR_SIGLA][CONFERIR_CAP])

    c = bt.comparar(texto_existente, texto_baixado)

    print(f"projeto ... {c.palavras_a} palavras")
    print(f"baixado ... {c.palavras_b} palavras")
    print(f"similaridade {c.similaridade:.4f}   (limiar {LIMIAR})")
    print()

    if c.identico:
        print("✅ IDÊNTICOS — o texto do projeto é exatamente esta edição.")
    elif c.similaridade >= LIMIAR:
        print(f"✅ MESMA EDIÇÃO, com {len(c.diferencas)} diferença(s) pequena(s):")
        for tipo, a, b in c.diferencas[:10]:
            print(f"   [{tipo}]")
            print(f"     projeto: ...{a}...")
            print(f"     baixado: ...{b}...")
        if len(c.diferencas) > 10:
            print(f"   ... e mais {len(c.diferencas) - 10}")
    else:
        print("❌ EDIÇÃO DIFERENTE.")
        print(f"   Referência: WEB x KJV no mesmo trecho dá ~0,83 — ou seja,")
        print(f"   {c.similaridade:.2f} é compatível com outra tradução, não com ruído.")
        print("   Confira se URLS_CANDIDATAS aponta pra WEB Classic (eng-web-c).")
        for tipo, a, b in c.diferencas[:5]:
            print(f"   [{tipo}]")
            print(f"     projeto: ...{a}...")
            print(f"     baixado: ...{b}...")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 4/4 — SALVAR                                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

import json, hashlib
from datetime import datetime, timezone

saida = {
    "fonte": url_usada,
    "baixado_em": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    "livros": {
        sigla: {
            str(cap): [{"n": v.numero, "t": v.texto, "q": v.quebra} for v in versiculos]
            for cap, versiculos in sorted(capitulos.items())
        }
        for sigla, capitulos in livros_lidos.items()
    },
}

texto_json = json.dumps(saida, ensure_ascii=False, indent=1)
CAMINHO_SAIDA.write_text(texto_json, encoding="utf-8")

sha = hashlib.sha256(texto_json.encode()).hexdigest()
print(f"💾 {CAMINHO_SAIDA}")
print(f"   {len(texto_json)/1e6:.1f} MB")
print(f"   sha256 {sha[:16]}...")
print()
print("Este arquivo é IMUTÁVEL — o texto bíblico não muda. Se um dia ele")
print("mudar, foi acidente: o sha256 acima é o que prova.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✅ USAR — gerar o roteiro de qualquer capítulo                    ║
# ╚══════════════════════════════════════════════════════════════════╝

# Daqui pra frente, qualquer capítulo sai numa chamada. Exemplo:
SIGLA, CAPITULO = "John", 3

livro = bl.por_sigla(SIGLA)
roteiro = bt.gerar_roteiro(livros_lidos[SIGLA][CAPITULO])

print(f"{livro.nome_projeto(CAPITULO)}  ({len(roteiro.split())} palavras)")
print("─" * 60)
print(roteiro[:400] + ("..." if len(roteiro) > 400 else ""))
print("─" * 60)
print()
print("Pra salvar na pasta de um vídeo:")
print(f'  destino = BASE / "videos" / "{livro.nome_projeto(CAPITULO)}"')
print(f'  destino.mkdir(parents=True, exist_ok=True)')
print(f'  (destino / f"{livro.nome_projeto(CAPITULO)}_roteiro_versiculos.txt")\\')
print(f'      .write_text(roteiro, encoding="utf-8")')